# FlashAttention-2 (FA2) Python Tiling Simulation

This notebook demonstrates the algorithmic changes introduced in **FlashAttention-2** compared to standard attention and FlashAttention-1:
1. **Flipped Loop Order**: The outer loop iterates over blocks of $Q$, and the inner loop iterates over blocks of $K$ and $V$.
2. **Delayed Normalization**: Storing the un-normalized output accumulator in SRAM registers and doing the division by the cumulative sum $l$ only once at the end of the inner loop.

We will implement these in pure PyTorch and verify their mathematical equivalence.

In [1]:
import math
import torch

print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.10.0+cu128


## 1. Standard Self-Attention Baseline

In [2]:
def standard_attention(Q, K, V):
    """
    Standard naive attention.
    Q, K, V shapes: (B, H, N, d)
    """
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, N, N)
    attn_weights = torch.softmax(scores, dim=-1)                   # (B, H, N, N)
    output = torch.matmul(attn_weights, V)                          # (B, H, N, d)
    return output

## 2. FlashAttention-2 Python Tiling Simulation

### Flipped Loop Updates:
* In FA1, the outer loop was over $K, V$ blocks, and the inner loop was over $Q$ blocks. This meant the running output $O$ was loaded and written to global memory multiple times.
* In FA2, the outer loop is over $Q$ blocks, and the inner loop is over $K, V$ blocks. A block of $Q$ is loaded into SRAM registers, updated against all blocks of $K, V$, and written to HBM **only once** when the inner loop finishes.

### Delayed Normalization updates:
* Instead of updating $O_i$ with division at each step, we maintain the un-normalized accumulator $S_i$ in registers:
  $$S_i^{\text{new}} = e^{m_i - m_i^{\text{new}}} S_i + e^{\tilde{m} - m_i^{\text{new}}} \tilde{P}_{ij} V_j$$
  $$l_i^{\text{new}} = e^{m_i - m_i^{\text{new}}} l_i + e^{\tilde{m} - m_i^{\text{new}}} \tilde{l}_{ij}$$
* Normalization division is performed at the end of the inner loop:
  $$O_i = S_i / l_i$$

In [3]:
def flash_attention_v2_sim(Q, K, V, B_r=64, B_c=64):
    """
    Python simulation of FlashAttention-2 Forward Tiling Pass.
    Q, K, V shapes: (B, H, N, d)
    """
    B, H, N, d = Q.shape
    d_k = d
    O = torch.zeros_like(Q)
    
    for b in range(B):
        for h in range(H):
            Q_bh = Q[b, h]  # (N, d)
            K_bh = K[b, h]  # (N, d)
            V_bh = V[b, h]  # (N, d)
            
            # Memory structures representing HBM storage
            O_bh = torch.zeros((N, d), device=Q.device, dtype=Q.dtype)
            
            Tr = math.ceil(N / B_r)
            Tc = math.ceil(N / B_c)
            
            # --- OUTER LOOP: Iterates over blocks of Q (Rows) ---
            for i in range(Tr):
                start_r = i * B_r
                end_r = min(start_r + B_r, N)
                
                Q_i = Q_bh[start_r:end_r, :]  # (B_r, d) - Loaded to SRAM registers
                
                # Initialize row registers for this Q_i block
                # Accumulator S_i initialized to 0, row max m_i to -inf, row sum l_i to 0
                S_i = torch.zeros((end_r - start_r, d), device=Q.device, dtype=Q.dtype)
                m_i = torch.full((end_r - start_r, 1), float('-inf'), device=Q.device, dtype=Q.dtype)
                l_i = torch.zeros((end_r - start_r, 1), device=Q.device, dtype=Q.dtype)
                
                # --- INNER LOOP: Iterates over blocks of K, V (Columns) ---
                for j in range(Tc):
                    start_c = j * B_c
                    end_c = min(start_c + B_c, N)
                    
                    K_j = K_bh[start_c:end_c, :]  # (B_c, d) - Loaded to SRAM
                    V_j = V_bh[start_c:end_c, :]  # (B_c, d) - Loaded to SRAM
                    
                    # Compute scores block: (B_r, B_c)
                    scores_ij = torch.matmul(Q_i, K_j.transpose(-2, -1)) / math.sqrt(d_k)
                    
                    # Local statistics
                    tilde_m_ij, _ = torch.max(scores_ij, dim=-1, keepdim=True)       # (B_r, 1)
                    tilde_P_ij = torch.exp(scores_ij - tilde_m_ij)                   # (B_r, B_c)
                    tilde_l_ij = torch.sum(tilde_P_ij, dim=-1, keepdim=True)         # (B_r, 1)
                    
                    # Update scaling statistics
                    m_i_new = torch.max(m_i, tilde_m_ij)
                    
                    alpha = torch.exp(m_i - m_i_new)  # Rescale running max
                    beta = torch.exp(tilde_m_ij - m_i_new)  # Rescale local max
                    
                    # Update sum
                    l_i = alpha * l_i + beta * tilde_l_ij
                    
                    # Update accumulator (no division operations here!)
                    S_i = alpha * S_i + beta * torch.matmul(tilde_P_ij, V_j)
                    
                    # Update running maximum
                    m_i = m_i_new
                
                # --- Post-Inner-Loop: Single division per block to write final output to HBM ---
                O_bh[start_r:end_r, :] = S_i / l_i
            
            O[b, h] = O_bh
            
    return O

## 3. Mathematical Verification

Let's verify that standard attention and the FlashAttention-2 simulation yield numerically identical outputs.

In [4]:
batch_size = 4
num_heads = 8
seq_len = 512
head_dim = 64

# Set seed for reproducibility
torch.manual_seed(42)

Q = torch.randn(batch_size, num_heads, seq_len, head_dim)
K = torch.randn(batch_size, num_heads, seq_len, head_dim)
V = torch.randn(batch_size, num_heads, seq_len, head_dim)

# Compute outputs
output_standard = standard_attention(Q, K, V)
output_flash2 = flash_attention_v2_sim(Q, K, V, B_r=64, B_c=64)

# Verify correctness
match = torch.allclose(output_standard, output_flash2, atol=1e-4, rtol=1e-4)
print(f"Standard Attention vs. FlashAttention-2 Simulation Match: {match}")

difference = torch.norm(output_standard - output_flash2).item()
print(f"Frobenius Norm of Difference: {difference:.4e}")

Standard Attention vs. FlashAttention-2 Simulation Match: True
Frobenius Norm of Difference: 2.5593e-05


## 4. Analysis: HBM Write Reductions

Let's compare the HBM writes for FlashAttention-1 and FlashAttention-2:

1. **FlashAttention-1**:
   * Loop order: outer loop over columns, inner loop over rows.
   * For each block of $K, V$ ($T_c$), we loop through all blocks of $Q$ ($T_r$).
   * At every step of the inner loop, we load $O_i$, perform intermediate updates (which include division), and write the updated $O_i$ block back to HBM.
   * Total output writes to HBM scale as $O(T_c \cdot T_r)$ which is quadratic in sequence length.

2. **FlashAttention-2**:
   * Loop order: outer loop over rows ($T_r$), inner loop over columns ($T_c$).
   * For each block of $Q$ ($T_r$), we load it to registers once, run the inner loop over all columns of $K, V$, and keep accumulation in fast SRAM registers.
   * We write the output block to HBM **only once** at the end of the inner loop.
   * Total output writes to HBM scale as $O(T_r)$ which is **strictly linear** in sequence length.

This loop restructure drastically reduces GPU global memory write traffic, leading to massive speedups in real-world implementations.

In [5]:
B = batch_size
H = num_heads
N = seq_len
d = head_dim
bytes_per_elem = 2  # FP16

# 1. Memory Footprint
input_size = B * H * N * d * bytes_per_elem / (1024 * 1024)   # MB
output_size = input_size                                      # MB
total_inputs_size = 3 * input_size                            # MB
intermediate_matrix_size = B * H * N * N * bytes_per_elem / (1024 * 1024)  # MB

# 2. FLOPs Calculations
flops_qk = B * H * 2 * (N ** 2) * d                           # FLOPs
flops_softmax = B * H * N * (5 * N - 2)                       # FLOPs
flops_av = B * H * 2 * (N ** 2) * d                            # FLOPs
total_flops = flops_qk + flops_softmax + flops_av             # FLOPs

# 3. HBM Memory Traffic Transfers (MB)
hbm_standard = total_inputs_size + 4 * intermediate_matrix_size + output_size
hbm_flash = total_inputs_size + output_size + (input_size * 0.75) # Inputs + output + reload overhead

# 4. Arithmetic Intensities (FLOPs/Byte)
intensity_standard = total_flops / (hbm_standard * 1024 * 1024)
intensity_flash = total_flops / (hbm_flash * 1024 * 1024)

# 5. NVIDIA T4 GPU Limits
t4_compute_peak = 65.0 * 1e12  # 65 TFLOPs in FLOP/s
t4_bandwidth_peak = 320.0 * 1e9  # 320 GB/s in B/s
t4_ridge_point = t4_compute_peak / t4_bandwidth_peak

# 6. Bound Execution Times (ms)
est_time_standard = (hbm_standard * 1024 * 1024) / t4_bandwidth_peak * 1000
est_time_flash = total_flops / t4_compute_peak * 1000

print("=" * 70)
print(f"COMPUTATION RESULTS FOR SHAPE (B={B}, H={H}, N={N}, d={d})")
print("=" * 70)
print(f"1. DATA SIZES:")
print(f"   - Single Input Q/K/V: {input_size:.2f} MB")
print(f"   - Total Inputs:       {total_inputs_size:.2f} MB")
print(f"   - Intermediate Matrix (scores/weights): {intermediate_matrix_size:.2f} MB")

print(f"\n2. COMPUTATIONAL FLOPS:")
print(f"   - Q @ K^T:            {flops_qk / 1e9:.4f} GFLOPs")
print(f"   - Softmax overhead:   {flops_softmax / 1e9:.4f} GFLOPs")
print(f"   - weights @ V:        {flops_av / 1e9:.4f} GFLOPs")
print(f"   - Total Compute:      {total_flops / 1e9:.4f} GFLOPs")

print(f"\n3. HBM MEMORY TRAFFIC (BYTES LOADED/WRITTEN):")
print(f"   - Standard Attention: {hbm_standard:.2f} MB")
print(f"   - Flash Attention:    {hbm_flash:.2f} MB (~{hbm_standard / hbm_flash:.1f}x reduction)")

print(f"\n4. ARITHMETIC INTENSITIES (FLOPs / Byte):")
print(f"   - Standard Attention: {intensity_standard:.2f} FLOPs/Byte")
print(f"   - Flash Attention:    {intensity_flash:.2f} FLOPs/Byte")
print(f"   - T4 GPU Ridge Point: {t4_ridge_point:.2f} FLOPs/Byte")

print(f"\n5. PERFORMANCE CLASSIFICATION (NVIDIA T4 GPU):")
is_std_mem = intensity_standard < t4_ridge_point
is_flash_mem = intensity_flash < t4_ridge_point
print(f"   - Standard Attention: {'Memory-Bound' if is_std_mem else 'Compute-Bound'} (Intensity {intensity_standard:.1f} vs Ridge Point {t4_ridge_point:.1f})")
print(f"   - Flash Attention:    {'Memory-Bound' if is_flash_mem else 'Compute-Bound'} (Intensity {intensity_flash:.1f} vs Ridge Point {t4_ridge_point:.1f})")
print(f"   - Standard Time Limit (Bandwidth Bound): {est_time_standard:.4f} ms")
print(f"   - Flash Time Limit (Compute Bound):      {est_time_flash:.4f} ms (~{est_time_standard / est_time_flash:.1f}x speedup potential)")
print("=" * 70)

COMPUTATION RESULTS FOR SHAPE (B=4, H=8, N=512, d=64)
1. DATA SIZES:
   - Single Input Q/K/V: 2.00 MB
   - Total Inputs:       6.00 MB
   - Intermediate Matrix (scores/weights): 16.00 MB

2. COMPUTATIONAL FLOPS:
   - Q @ K^T:            1.0737 GFLOPs
   - Softmax overhead:   0.0419 GFLOPs
   - weights @ V:        1.0737 GFLOPs
   - Total Compute:      2.1894 GFLOPs

3. HBM MEMORY TRAFFIC (BYTES LOADED/WRITTEN):
   - Standard Attention: 72.00 MB
   - Flash Attention:    9.50 MB (~7.6x reduction)

4. ARITHMETIC INTENSITIES (FLOPs / Byte):
   - Standard Attention: 29.00 FLOPs/Byte
   - Flash Attention:    219.79 FLOPs/Byte
   - T4 GPU Ridge Point: 203.12 FLOPs/Byte

5. PERFORMANCE CLASSIFICATION (NVIDIA T4 GPU):
   - Standard Attention: Memory-Bound (Intensity 29.0 vs Ridge Point 203.1)
   - Flash Attention:    Compute-Bound (Intensity 219.8 vs Ridge Point 203.1)
   - Standard Time Limit (Bandwidth Bound): 0.2359 ms
   - Flash Time Limit (Compute Bound):      0.0337 ms (~7.0x speedup pote